# Are Sisters, or Cousins, Morphologically More Alike Than Random? (vem 1uM, 144hrs Pre-Treatment)

`vem_1uM_144hrsbefore` imaging starts 6 days before 1uM vemurafenib is
added (frame 7 is day 0; frame 6 - the day before, day -1 - is
currently missing entirely from this dataset). Frames 1-6 are
therefore pre-treatment (effectively naive growth, used here as a
**control**); frames 7-11 are post-treatment (the **treatment**
range). Both ranges' end frames are notebook-level parameters
(`config.CONTROL_END_FRAME`, `config.TREATMENT_END_FRAME`).

**Question**: is a cell's morphology more similar to its sister's (born
from the same division) than to a random cell's? What about its
cousins' (its parent and its parent's sister's children - i.e. sharing
a grandparent, not a parent)? This notebook answers both, separately
for the control and treatment ranges, so the two can be compared: does
vemurafenib change how (dis)similar relatives are, relative to how
things looked before treatment?

Sister/cousin relationships are reconstructed directly from this
dataset's full lineage tracking (`node_id`/`pred_node_id`/`lin_id` -
see `shared/lineage_tree.py`), **not** from `lineage_frame7_to_frame1.mat`
(that file only resolves the single frame-1 ancestor of each day-0
lineage, collapsing every division in between - the generation-by-
generation structure used here needs more than that).

Requires `main.py` to have already been run (for `selected_metadata.csv`
and `{cell,nuc}_qgw_075P_dists.csv` under `output/cajal_output/`).


In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/06_vem_144hr_lineage_relatedness/notebooks/), so the analysis
# dir - and config.py - is one level up, and the repo root - where
# shared/ lives - is two more levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR/CAJAL_OUTPUT_DIR are relative to the analysis
# directory, not this notebook's own cwd - re-anchor them to an absolute
# path once, here
config.OUTPUT_DIR       = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.CAJAL_OUTPUT_DIR)

from shared.cajal.embed import load_distance_matrix
from shared.load_data import parse_label
from shared.lineage_tree import build_segments, find_sister_pairs, find_cousin_pairs
from shared.pair_similarity import pair_distance_permutation_test, plot_pair_similarity_test


## Method

1. **Reconstruct every lineage tree** in frames 1-`config.TREATMENT_END_FRAME`
   from the FULL metadata (not just the cells CAJAL actually ran on -
   see below), and find every sister pair and cousin pair in it
   (`shared/lineage_tree.py`).
2. **For each modality** (cell, nucleus) **and each range** (control,
   treatment):
   - Restrict each pair's two sides to whichever of their own frames
     fall in this range AND actually got a CAJAL embedding (survived
     `main.py`'s subsampling - a simple, lineage-unaware subsample, so
     which true pairs keep both members is left to chance). Use every
     surviving (frame, pixel_id) combination on each side as its own
     real pair - sisters and cousins alike are **not** required to
     share a frame (a sister's own track can run longer than its
     sibling's; a cousin pair's parents can divide at different times).
   - Build ONE shared null distribution: many random pairs, drawn (with
     replacement) from every CAJAL-embedded cell in this same range,
     any frame - not frame-matched, same as the real pairs above.
   - Compare the real sister-pair mean distance, and separately the
     real cousin-pair mean distance, against that SAME null
     (`shared/pair_similarity.py`'s `pair_distance_permutation_test` -
     called twice with the same pool/draw-size/seed, which is what
     makes the null identical both times).
3. **Plot**: one figure per (modality, range) - the shared null
   distribution with both the sister-pair and cousin-pair observed
   means marked (`plot_pair_similarity_test`).


In [ ]:
PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '01_relatedness')
SVG_DIR = os.path.join(PLOTS_DIR, 'svgs')
MODALITY_NAMES = {'cell': 'cell', 'nuc': 'nucleus'}
os.makedirs(SVG_DIR, exist_ok=True)

# (start, end) frame, inclusive - 'control' is pre-treatment, 'treatment'
# is post-treatment (config.TREATMENT_START_FRAME is fixed by the data -
# when the drug was actually added - config.CONTROL_END_FRAME/
# TREATMENT_END_FRAME are this notebook's own tunable range limits)
FRAME_RANGES = {
    'control':   (1, config.CONTROL_END_FRAME),
    'treatment': (config.TREATMENT_START_FRAME, config.TREATMENT_END_FRAME),
}


def observations_in_range(segment, frame_range, label_to_index):
    '''
    A segment's own (frame, pixel_id) observations, restricted to a
    frame range AND to whichever ones actually got a CAJAL embedding -
    returned as indices into the distance matrix label_to_index was
    built from.
    '''
    lo, hi = frame_range
    return [
        label_to_index[(frame, pixel_id)]
        for frame, pixel_id in segment.observations
        if lo <= frame <= hi and (frame, pixel_id) in label_to_index
    ]


def real_index_pairs(pairs, frame_range, label_to_index):
    '''
    Every (index_a, index_b) cross-combination of a surviving
    observation from one side of a sister/cousin pair with a surviving
    observation from the other - every combination available, not just
    one per pair (see this notebook's "Method" above).
    '''
    index_pairs = []
    for segment_a, segment_b in pairs:
        idx_a = observations_in_range(segment_a, frame_range, label_to_index)
        idx_b = observations_in_range(segment_b, frame_range, label_to_index)
        index_pairs.extend((i, j) for i in idx_a for j in idx_b)
    return index_pairs


def pool_in_range(label_to_index, frame_range):
    '''Every CAJAL-embedded cell (any lineage relationship) within a frame range - the null is drawn from this.'''
    lo, hi = frame_range
    return [idx for (frame, pixel_id), idx in label_to_index.items() if lo <= frame <= hi]


def process_modality(modality, segments):
    '''
    For one modality: load its real distance matrix, restrict the
    sister/cousin pairs found in `segments` to each frame range in
    turn, and run+plot the shared-null permutation test for each.

    Inputs:
        modality: 'cell' or 'nuc'
        segments: from build_segments, across every lineage tree in
                  frames 1-config.TREATMENT_END_FRAME
    '''
    sister_pairs = find_sister_pairs(segments)
    cousin_pairs = find_cousin_pairs(segments)
    print(f"[{modality}] {len(sister_pairs)} total sister pair(s), {len(cousin_pairs)} total cousin pair(s) across every lineage tree")

    dist_csv = os.path.join(config.CAJAL_OUTPUT_DIR, f'{modality}_qgw_{config.N_POINTS:03d}P_dists.csv')
    labels, dist_matrix = load_distance_matrix(dist_csv)
    label_to_index = {}
    for i, label in enumerate(labels):
        _, frame, pixel_id = parse_label(label)
        label_to_index[(frame, pixel_id)] = i

    for range_name, frame_range in FRAME_RANGES.items():
        real_sister_idx = real_index_pairs(sister_pairs, frame_range, label_to_index)
        real_cousin_idx = real_index_pairs(cousin_pairs, frame_range, label_to_index)
        pool = pool_in_range(label_to_index, frame_range)

        print(f"  [{range_name}, frames {frame_range[0]}-{frame_range[1]}] "
              f"{len(real_sister_idx)} sister-pair observation(s), {len(real_cousin_idx)} cousin-pair observation(s), "
              f"{len(pool)} cell(s) in the pool")

        if len(real_sister_idx) == 0 or len(real_cousin_idx) == 0:
            print(f"    Skipping: need at least one sister-pair AND one cousin-pair observation with both sides sampled")
            continue

        # both tests share this SAME draw size, pool, seed, and
        # n_permutations, which is what makes their null distributions
        # identical (see pair_distance_permutation_test's own docstring)
        n_pairs_per_draw = min(len(real_sister_idx), len(real_cousin_idx))

        sister_mean, null_means, sister_p = pair_distance_permutation_test(
            dist_matrix, pool, real_sister_idx, n_pairs_per_draw,
            n_permutations=config.N_PERMUTATIONS, seed=config.TEST_SEED,
        )
        cousin_mean, null_means_check, cousin_p = pair_distance_permutation_test(
            dist_matrix, pool, real_cousin_idx, n_pairs_per_draw,
            n_permutations=config.N_PERMUTATIONS, seed=config.TEST_SEED,
        )
        assert np.array_equal(null_means, null_means_check), "the two calls' null distributions must be identical"

        print(f"    sisters: mean distance = {sister_mean:.4f} (p = {sister_p:.4f})")
        print(f"    cousins: mean distance = {cousin_mean:.4f} (p = {cousin_p:.4f})")

        fig = plot_pair_similarity_test(null_means, {
            'sisters': (sister_mean, sister_p),
            'cousins': (cousin_mean, cousin_p),
        }, title=f'{MODALITY_NAMES[modality]} morphology: {range_name} (frames {frame_range[0]}-{frame_range[1]})')
        fig.savefig(os.path.join(PLOTS_DIR, f'{modality}_{range_name}.pdf'), bbox_inches='tight')
        fig.savefig(os.path.join(SVG_DIR, f'{modality}_{range_name}.svg'), bbox_inches='tight')
        plt.close(fig)
        print(f"    Saved to {PLOTS_DIR}")

        yield {
            'modality': modality, 'frame_range': range_name,
            'null_distance': null_means,
            'sister_mean_distance': sister_mean, 'sister_p_value': sister_p,
            'cousin_mean_distance': cousin_mean, 'cousin_p_value': cousin_p,
        }


## Build the lineage tree (once - shared across both modalities)

In [ ]:
full_metadata = pd.read_csv(config.metadata_csv_path())

# tree-building uses the FULL metadata (every tracked cell, has_mask
# True or False) - has_mask only reflects whether a usable mask crop
# could be extracted, not whether the cell/lineage tracking itself
# exists. Filtering to has_mask=True here would drop intermediate rows
# a later frame's pred_node_id still needs, breaking chains that are
# otherwise intact.
full_metadata = full_metadata[full_metadata['frame'] <= config.TREATMENT_END_FRAME].reset_index(drop=True)

segments = build_segments(full_metadata)
print(f"Built {len(segments)} segment(s) across every lineage tree in frames 1-{config.TREATMENT_END_FRAME}")


## Run: cell morphology

In [ ]:
cell_results = list(process_modality('cell', segments))


## Run: nuclear morphology

In [ ]:
nuc_results = list(process_modality('nuc', segments))


## Export plot data

Combines every (modality, frame_range) result above into one CSV
(`plot_data.csv`, alongside the PDFs/SVGs above): one row per
permutation draw, with `sister_mean_distance`/`sister_p_value`/
`cousin_mean_distance`/`cousin_p_value` repeated on every row of their
(modality, frame_range) group, so each figure's full null histogram and
both observed markers can be redrawn from this one flat file.

In [ ]:
plot_data = pd.concat([
    pd.DataFrame({
        'modality': r['modality'], 'frame_range': r['frame_range'],
        'null_distance': r['null_distance'],
        'sister_mean_distance': r['sister_mean_distance'], 'sister_p_value': r['sister_p_value'],
        'cousin_mean_distance': r['cousin_mean_distance'], 'cousin_p_value': r['cousin_p_value'],
    })
    for r in (cell_results + nuc_results)
], ignore_index=True)

plot_data_path = os.path.join(PLOTS_DIR, 'plot_data.csv')
plot_data.to_csv(plot_data_path, index=False)
print(f"  Saved plot data ({len(plot_data)} rows) to {plot_data_path}")
